In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from keras import optimizers
from keras.layers import LSTM
from keras.layers import Dense
from keras.models import Sequential
processed_dpath = '/Users/oxide/Documents/research/orenstein/code/P288FinalProject/processed_data/'

In [ ]:
##
## LSTM goal: predict erruption activity over the next predict_window minutes based
##            on erruption activity and other measurements over the last train_window minutes
##              
##            the input to this segment is dataInterpolated.csv, which has all the compiled data that we want to analysze
##

# define some stuff
predict_window = 1*24*60 # 1 day
train_window = 7*24*60 # 1 week

# Markus function for numpy array
def prepare_data(data, n_past, n_future):
    x, y = [], []
    for i in range(n_past, len(data) - n_future + 1):
        x.append(data[i - n_past:i])
        y.append(data[i:i + n_future])
    return np.array(x), np.array(y)

def PlotData(groups, wind = 'No', values = None):
    
    if (values is None):
        dataset = pd.read_csv('pollution.csv', header=0, index_col=0)
        values = dataset.values
        cols   = dataset.columns
    else:
        cols   = values.columns
        
            
    if wind != 'No':
        #just to check for plt
        WindSpdTot     = values['wnd_spd [SE]'] + values['wnd_spd [NE]'] +\
                         values['wnd_spd [NW]'] + values['wnd_spd [cv]']
            
    values = np.array(values)#making sure we have the right format for plotting
    
    
    i = 1
    # plot each column
    x = np.arange(0, len(values[:, 0]),1)
    x = x/24.0 #hrs --> days
    plt.figure(figsize = (10, 2*len(groups)))
    # specify columns to plot
    for group in groups:
        plt.subplot(len(groups), 1, i,)
        #plotting one-month moving average
        y_smooth = savgol_filter(values[:,group], 24*7*4, 5)
        plt.plot(x, y_smooth, c = 'black')
        plt.plot(x, values[:, group], c = [215/255, 0, 64/255, 0.5])
        if wind != 'No':
            if group in [4, 5, 6, 7]:
                #plotting total wind speed for comparison
                plt.plot(x, WindSpdTot, c = [0, 71/255, 171/255, 0.2])
        plt.title(cols[group], y = 0.75, loc = 'right', fontsize = 12)
        i += 1
    plt.xlabel('time [days]')
    plt.show()

def DoTimeLag(Data, dt):
    
    Y = Data[dt:, 0]
    Y = Y.reshape((len(Y), 1))
    X = Data[dt:, 1:]
    
    for i in reversed(range(dt)):
        X = concatenate((X, Data[i:-dt+i, 1:]), axis = 1)
        
    Data = concatenate((Y,X), axis = 1)
        
    return Data

def NormAndScale(dataframe):
    """
    normalize data to [0, 1] and keep original copy
    """
    
    # extracting values from dataframe
    df_scaled = pd.copy(dataframe)
    data = df.to_numpy()
    
    # normalize
    scaler = MinMaxScaler(feature_range=(0, 1)) #normalize features
    scaled = scaler.fit_transform(data).astype('float32')
    
    # storing scaled values into a dataframe
    df_scaled[:] = scaled
        
    return(df_scaled, scaler)

def invNormAndScale(X, model, scaler):
    
    Yhat     = model.predict(X)
    
    X        = X[:,0,:]
    
    #merging wind direction
    Wind     = np.sum(X[:,4:8], axis = 1)
    Wind     = Wind.reshape((len(Wind),1))
    
    X_new    = np.hstack((X[:,:4], Wind, X[:,8:]))
    
    #invert scaling for forecast
    inv_yhat = concatenate((Yhat, X_new), axis = 1)
    inv_yhat = scaler.inverse_transform(inv_yhat)
    inv_yhat = inv_yhat[:,0]
    
    return(inv_yhat)

class LSTM_keras():

    def __init__(self, lstm_vars =['Erruption Activity', 'VPCC RSAM', 'VPPC RSAM', 'VPNC RSAM', 'VPRS RSAM', 'VPRS Field E', 'VPRS Field N', 'VPRS Field Z', 'CO2 Concentration']):
        """
        lstm_vars: what variables to actually use in the LSTM
        """
        
        # read in dataInterpolated.csv
        df = pd.read_csv(processed_dpath+f'dataInterpolated.csv', index_col=0)

        # filter data to columns we want in LSTM and handle any nans
        df = data[lstm_vars]
        df.fillna(0, inplace=True)
        self.df = df # save DataFrame
        
        # preprocess data into numpy arrays
        [self.df_scaled self.scaler] = NormAndScale(df)
        self.data = self.df_scaled.to_numpy()

    def CreateModel(self, n_divide = 80, n_neurons = 800, dt = 1,\
                          n_epochs = 150, learning_rate = 0.001,\
                          momentum = 0.4, opt = 'adam'):
        
        print('Performing Training...')
        
        #calling normalized and scaled data
        Data = self.Data
        
        if dt > 1:#multiple time lag is treated as dt times extra features
            Data = DoTimeLag(Data, dt - 1)
        
        n_divide = round(len(Data[:,0])*n_divide/100)
        
        #dividing data into test and training set
        train = Data[:n_divide, :]
        test  = Data[n_divide:, :]
        
        #split into input and outputs
        train_X, train_Y = train[:, 1:], train[:, 0]
        test_X,  test_Y  = test[:, 1:], test[:, 0]
        
        #reshape input to be 3D [samples, timesteps, features]
        train_X = train_X.reshape((train_X.shape[0], dt,\
                                   int(train_X.shape[1]/dt)))
        test_X  = test_X.reshape((test_X.shape[0], dt,\
                                  int(test_X.shape[1]/dt)))
            
        print(train_X.shape, train_Y.shape, test_X.shape, test_Y.shape)
 
        #design network: n_neurons stands for size of hidden layer
        model = Sequential()
        model.add(LSTM(n_neurons, input_shape = (train_X.shape[1], train_X.shape[2])))
        model.add(Dense(1))
        
        if opt == 'sgd':
            Opt = optimizers.SGD(learning_rate = learning_rate,\
                                 momentum = momentum)
        
        if opt == 'adam':
            Opt = optimizers.Adam(learning_rate = learning_rate)
            
        
        model.compile(loss = 'mae', optimizer = Opt)
        # fit network
        history = model.fit(train_X, train_Y, epochs = n_epochs,\
                            batch_size = 24*14,\
                            validation_data = (test_X, test_Y),\
                            verbose = 2, shuffle = False)
        #plot history
        plt.plot(history.history['loss'], label='train')
        plt.plot(history.history['val_loss'], label='test')
        plt.xlabel('epoch')
        plt.ylabel('loss [MSE]')
        plt.legend()
        plt.show()
        
        print('...Training Done!')
        
        self.dt       = dt
        self.model    = model
        self.test_X   = test_X
        self.test_Y   = test_Y
        self.train_X  = train_X
        self.train_Y  = train_Y
        self.n_divide = n_divide
        
        
    def Predict(self):
        
        YOrig    = self.OrigData[:,0]
        
        #scaled X we need for the inverse scaler
        #make sure to have correct shape, if dt > 1
        test_X       = self.test_X
        train_X      = self.train_X
        
        #predicing and rescaling data
        Yhat_test  = invNormAndScale(test_X, self.model, self.scaler)
        Yhat_train = invNormAndScale(train_X, self.model, self.scaler)
        
        #calculate RMSE
        #rmse = sqrt(mean_squared_error(YOrig[self.dt-1+self.n_divide:],\
        #                               Yhat_test))
        #print('Test RMSE: %.3f' % rmse)
        
        fittedY = np.hstack((Yhat_train, Yhat_test))       
        All     = np.hstack((fittedY, YOrig))
        
        
        #plotting final result: actual data vs prediction (training and test)
        xo = np.arange(0, len(YOrig), 1)
        xo = xo/24.0 #hrs --> days
        
        xf = np.arange(0, len(fittedY), 1)
        xf = xf/24.0 #hrs --> days
        
        M = np.max(All)
        m = np.min(All)
        
        plt.plot(xf, fittedY, c = [215/255, 0, 64/255, 0.8])
        plt.plot(xo, YOrig,   c = [0, 71/255, 171/255, 0.2])
        plt.xlabel('time [days]')
        plt.ylabel('pollution')
        plt.legend(['$\hat{y}$','y'], loc = 'upper left')
        plt.fill_between([self.n_divide/24, len(fittedY)/24], m, M, color = 'k',\
                         alpha = 0.1)
        plt.title('prediction', y = 0.75, loc = 'right', fontsize = 12)
        plt.show()
        
        #weekly moving average
        yf  = savgol_filter(fittedY,24*7*4,5)
        yo  = savgol_filter(YOrig,24*7*4,5)
        All = np.hstack((yf, yo))
        M   = np.max(All)
        m   = np.min(All)
        plt.plot(xf, yf, c = [215/255, 0, 64/255, 0.8])
        plt.plot(xo, yo, c = [0, 71/255, 171/255, 0.2])
        plt.xlabel('time [days]')
        plt.ylabel('pollution [monthly avg]')
        plt.legend(['$\hat{y}$','y'], loc = 'upper left')
        plt.fill_between([self.n_divide/24, len(fittedY)/24], m, M, color = 'k',\
                         alpha = 0.1)
        plt.title('prediction', y = 0.75, loc = 'right', fontsize = 12)
        plt.show()

In [15]:
# load data, keep only certain variables, and fill Nans
vars = ['Erruption Activity', 'VPCC RSAM', 'VPPC RSAM', 'VPNC RSAM', 'VPRS RSAM', 'VPRS Field E', 'VPRS Field N', 'VPRS Field Z', 'CO2 Concentration']
data = pd.read_csv(processed_dpath+f'dataInterpolated.csv', index_col=0)
data = data[vars]
data.fillna(0, inplace=True)
data.name = 'lstmData'
data.to_csv(processed_dpath+f'{data.name}.csv')

# normalize features
values = data.values
scaler = MinMaxScaler(feature_range=(0, 1))
scaled = scaler.fit_transform(values)

# reframe as supervised learning
reframed = series_to_supervised(scaled, 1, 1)



In [16]:
reframed

,var1(t-1),var2(t-1),var3(t-1),var4(t-1),var5(t-1),var6(t-1),var7(t-1),var8(t-1),var9(t-1),var1(t),var2(t),var3(t),var4(t),var5(t),var6(t),var7(t),var8(t),var9(t)
1,0.004153,0.003712,0.015309,0.008843,0.008848,0.457519,0.742275,0.000185,0.384938,0.004153,0.003618,0.015622,0.008825,0.008762,0.456156,0.742275,0.000185,0.387442
2,0.004153,0.003618,0.015622,0.008825,0.008762,0.456156,0.742275,0.000185,0.387442,0.004153,0.003525,0.015935,0.008807,0.008677,0.455248,0.742626,0.000185,0.389947
3,0.004153,0.003525,0.015935,0.008807,0.008677,0.455248,0.742626,0.000185,0.389947,0.004153,0.003432,0.016248,0.008789,0.008591,0.453657,0.743329,0.000185,0.393135
4,0.004153,0.003432,0.016248,0.008789,0.008591,0.453657,0.743329,0.000185,0.393135,0.004153,0.003338,0.016560,0.008771,0.008506,0.450477,0.742626,0.000185,0.393819
5,0.004153,0.003338,0.016560,0.008771,0.008506,0.450477,0.742626,0.000185,0.393819,0.004153,0.003245,0.016873,0.008754,0.008420,0.446842,0.741924,0.000246,0.394503
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76316,0.004153,0.007373,0.168550,0.069996,0.004530,0.337119,0.586728,0.999508,0.997770,0.004153,0.006873,0.169363,0.069764,0.004451,0.336665,0.584972,0.999631,0.998216
76317,0.004153,0.006873,0.169363,0.069764,0.004451,0.336665,0.584972,0.999631,0.998216,0.004153,0.006372,0.170176,0.069531,0.004371,0.335075,0.583919,0.999692,0.998662
76318,0.004153,0.006372,0.170176,0.069531,0.004371,0.335075,0.583919,0.999692,0.998662,0.004153,0.005871,0.170990,0.069298,0.004291,0.334393,0.582514,0.999815,0.999108
76319,0.004153,0.005871,0.170990,0.069298,0.004291,0.334393,0.582514,0.999815,0.999108,0.004153,0.005370,0.171803,0.069065,0.004211,0.334166,0.581812,0.999938,0.999554
